March Madness / college basketball prediction project centered on building a model that predicts NCAA tournament games from historical team-level data.

In [1]:
import numpy as np
import pandas as pd
import os

In [2]:
CURRENT_SEASON = 2026
DATA_DIR = 'march-machine-learning-mania-2026'

In [3]:
csv_files = [f for f in os.listdir(DATA_DIR) if f.endswith('.csv')]
data = {os.path.splitext(f)[0]: pd.read_csv(os.path.join(DATA_DIR, f)) for f in sorted(csv_files)}

for name, df in data.items():
    print(f"{name}: {df.shape}")

Cities: (509, 3)
Conferences: (51, 2)
MConferenceTourneyGames: (6793, 5)
MGameCities: (91940, 6)
MMasseyOrdinals: (5819228, 5)
MNCAATourneyCompactResults: (2585, 8)
MNCAATourneyDetailedResults: (1449, 34)
MNCAATourneySeedRoundSlots: (776, 5)
MNCAATourneySeeds: (2626, 3)
MNCAATourneySlots: (2586, 4)
MRegularSeasonCompactResults: (198079, 8)
MRegularSeasonDetailedResults: (124031, 34)
MSeasons: (42, 6)
MSecondaryTourneyCompactResults: (1865, 9)
MSecondaryTourneyTeams: (1895, 3)
MTeamCoaches: (13900, 5)
MTeamConferences: (13753, 3)
MTeamSpellings: (1178, 2)
MTeams: (381, 4)
SampleSubmissionStage1: (519144, 2)
SampleSubmissionStage2: (132133, 2)
WConferenceTourneyGames: (6481, 5)
WGameCities: (88621, 6)
WNCAATourneyCompactResults: (1717, 8)
WNCAATourneyDetailedResults: (961, 34)
WNCAATourneySeeds: (1744, 3)
WNCAATourneySlots: (1780, 4)
WRegularSeasonCompactResults: (142093, 8)
WRegularSeasonDetailedResults: (86773, 34)
WSeasons: (29, 6)
WSecondaryTourneyCompactResults: (906, 9)
WSecondaryT

In [4]:
display(data['MRegularSeasonDetailedResults'])

,Season,DayNum,WTeamID,WScore,LTeamID,LScore,WLoc,NumOT,WFGM,WFGA,...,LFGA3,LFTM,LFTA,LOR,LDR,LAst,LTO,LStl,LBlk,LPF
0,2003,10,1104,68,1328,62,N,0,27,58,...,10,16,22,10,22,8,18,9,2,20
1,2003,10,1272,70,1393,63,N,0,26,62,...,24,9,20,20,25,7,12,8,6,16
2,2003,11,1266,73,1437,61,N,0,24,58,...,26,14,23,31,22,9,12,2,5,23
3,2003,11,1296,56,1457,50,N,0,18,38,...,22,8,15,17,20,9,19,4,3,23
4,2003,11,1400,77,1208,71,N,0,30,61,...,16,17,27,21,15,12,10,7,1,14
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
124026,2026,118,1373,76,1351,61,H,0,24,53,...,22,7,9,6,25,12,6,3,7,15
124027,2026,118,1378,90,1408,62,H,0,30,67,...,28,18,27,7,22,13,4,3,18,18
124028,2026,118,1389,63,1265,56,A,0,24,49,...,19,8,12,14,18,10,11,4,11,17
124029,2026,118,1455,84,1427,67,A,0,30,74,...,17,20,28,13,24,3,5,8,6,14


In [5]:
print(data['MRegularSeasonDetailedResults'].columns)

Index(['Season', 'DayNum', 'WTeamID', 'WScore', 'LTeamID', 'LScore', 'WLoc',
       'NumOT', 'WFGM', 'WFGA', 'WFGM3', 'WFGA3', 'WFTM', 'WFTA', 'WOR', 'WDR',
       'WAst', 'WTO', 'WStl', 'WBlk', 'WPF', 'LFGM', 'LFGA', 'LFGM3', 'LFGA3',
       'LFTM', 'LFTA', 'LOR', 'LDR', 'LAst', 'LTO', 'LStl', 'LBlk', 'LPF'],
      dtype='str')


In [ ]:
BOX_STATS = ['Score', 'FGM', 'FGA', 'FGM3', 'FGA3', 'FTM', 'FTA',
             'OR', 'DR', 'Ast', 'TO', 'Stl', 'Blk', 'PF']

PRIOR_OFFENSIVE_EFFICIENCY = 100.0
PRIOR_DEFENSIVE_EFFICIENCY = 100.0
EFFICIENCY_LEARNING_RATE = 0.10
LEAGUE_PRIOR_GAMES = 200
AVERAGE_POSSESSIONS_PER_TEAM = 69.0
HOME_COURT_ADVANTAGE = 3.5  # points per 100 possessions
SEASON_CARRYOVER_WEIGHT = 0.5
RESIDUAL_CAP = 15.0

regular_season_results = data['MRegularSeasonDetailedResults'].copy()
regular_season_results['is_tournament'] = 0

tournament_results = data['MNCAATourneyDetailedResults'].copy()
tournament_results['is_tournament'] = 1

results = pd.concat([regular_season_results, tournament_results], ignore_index=True).sort_values(
    ['Season', 'DayNum']).reset_index(drop=True)

# {(season, team_id): {'sums': {stat: float}, 'game_count': int, ...}}
team_rolling = {}

# {season: {'total_points': float, 'total_possessions': float, 'game_count': int}}
league_totals = {}

def get_or_create_entry(season, team_id):
    key = (season, team_id)
    if key not in team_rolling:
        sums = {}
        for stat in BOX_STATS:
            sums[stat] = 0.0
            sums[f'opponent_{stat}'] = 0.0

        # Season-to-season carryover with regression to mean
        prior_key = (season - 1, team_id)
        if prior_key in team_rolling:
            prior = team_rolling[prior_key]
            initial_offensive = (SEASON_CARRYOVER_WEIGHT * prior['offensive_efficiency']
                                 + (1 - SEASON_CARRYOVER_WEIGHT) * PRIOR_OFFENSIVE_EFFICIENCY)
            initial_defensive = (SEASON_CARRYOVER_WEIGHT * prior['defensive_efficiency']
                                 + (1 - SEASON_CARRYOVER_WEIGHT) * PRIOR_DEFENSIVE_EFFICIENCY)
        else:
            initial_offensive = PRIOR_OFFENSIVE_EFFICIENCY
            initial_defensive = PRIOR_DEFENSIVE_EFFICIENCY

        team_rolling[key] = {
            'sums': sums,
            'game_count': 0,
            'offensive_efficiency': initial_offensive,
            'defensive_efficiency': initial_defensive,
            'total_possessions': 0.0,
        }
    return team_rolling[key]

def estimate_possessions(game, prefix):
    # FGA - OR + TO + 0.475 * FTA (standard Kenpom possession estimate)
    return game[f'{prefix}FGA'] - game[f'{prefix}OR'] + game[f'{prefix}TO'] + 0.475 * game[f'{prefix}FTA']

def get_or_create_league_totals(season):
    if season not in league_totals:
        league_totals[season] = {'total_points': 0.0, 'total_possessions': 0.0, 'game_count': 0}
    return league_totals[season]

def get_league_average_efficiency(season):
    totals = get_or_create_league_totals(season)

    phantom_games = max(0, LEAGUE_PRIOR_GAMES - totals['game_count'])
    if phantom_games == 0:
        return totals['total_points'] / totals['total_possessions'] * 100.0

    if season - 1 in league_totals and league_totals[season - 1]['total_possessions'] > 0:
        prior_efficiency = league_totals[season - 1]['total_points'] / league_totals[season - 1]['total_possessions'] * 100.0
    else:
        prior_efficiency = 100.0

    phantom_possessions = phantom_games * 2 * AVERAGE_POSSESSIONS_PER_TEAM
    phantom_points = phantom_possessions * prior_efficiency / 100.0

    blended_points = phantom_points + totals['total_points']
    blended_possessions = phantom_possessions + totals['total_possessions']
    return blended_points / blended_possessions * 100.0

def update_team_box_stats(entry, game, own_prefix, opp_prefix):
    entry['game_count'] += 1
    for stat in BOX_STATS:
        entry['sums'][stat] += game[f'{own_prefix}{stat}']
        entry['sums'][f'opponent_{stat}'] += game[f'{opp_prefix}{stat}']

def extract_team_features(entry):
    """Extract 40 features from a team's rolling entry."""
    game_count = entry['game_count']
    sums = entry['sums']
    features = {}

    # 28 per-game averages (14 own + 14 opponent)
    for stat in BOX_STATS:
        features[f'avg_{stat.lower()}'] = sums[stat] / game_count
        features[f'avg_opponent_{stat.lower()}'] = sums[f'opponent_{stat}'] / game_count

    # 8 rate stats (from sums for proper weighting)
    def safe_divide(numerator, denominator):
        return numerator / denominator if denominator > 0 else 0.0

    features['fg_pct'] = safe_divide(sums['FGM'], sums['FGA'])
    features['fg3_pct'] = safe_divide(sums['FGM3'], sums['FGA3'])
    features['ft_pct'] = safe_divide(sums['FTM'], sums['FTA'])
    features['fg3_rate'] = safe_divide(sums['FGA3'], sums['FGA'])
    features['opponent_fg_pct'] = safe_divide(sums['opponent_FGM'], sums['opponent_FGA'])
    features['opponent_fg3_pct'] = safe_divide(sums['opponent_FGM3'], sums['opponent_FGA3'])
    features['opponent_ft_pct'] = safe_divide(sums['opponent_FTM'], sums['opponent_FTA'])
    features['opponent_fg3_rate'] = safe_divide(sums['opponent_FGA3'], sums['opponent_FGA'])

    # 3 efficiency
    features['offensive_efficiency'] = entry['offensive_efficiency']
    features['defensive_efficiency'] = entry['defensive_efficiency']
    features['net_efficiency'] = entry['offensive_efficiency'] - entry['defensive_efficiency']

    # 1 tempo
    features['avg_possessions'] = entry['total_possessions'] / game_count

    return features

def featurize_game(team_a_entry, team_b_entry, season, day_number, team_a_id, team_b_id, label, team_a_location, is_tournament):
    """Build a feature row from two teams' rolling entries (team_a = lower ID).

    Returns None if either team has no games yet.
    """
    if team_a_entry['game_count'] == 0 or team_b_entry['game_count'] == 0:
        return None

    team_a_features = extract_team_features(team_a_entry)
    team_b_features = extract_team_features(team_b_entry)

    row = {}
    for key, value in team_a_features.items():
        row[f'team_a_{key}'] = value
    for key, value in team_b_features.items():
        row[f'team_b_{key}'] = value
    for key in team_a_features:
        row[f'diff_{key}'] = team_a_features[key] - team_b_features[key]

    row['season'] = season
    row['day_number'] = day_number
    row['team_a_id'] = team_a_id
    row['team_b_id'] = team_b_id
    row['label'] = label
    row['team_a_location'] = team_a_location
    row['is_tournament'] = is_tournament

    return row

training_rows = []

for _, game in results.iterrows():
    winner_entry = get_or_create_entry(game['Season'], game['WTeamID'])
    loser_entry = get_or_create_entry(game['Season'], game['LTeamID'])

    # Featurize both teams' rolling stats (pre-update) for training data
    winner_team_id = game['WTeamID']
    loser_team_id = game['LTeamID']
    if winner_team_id < loser_team_id:
        team_a_location = game['WLoc']
        row = featurize_game(winner_entry, loser_entry, game['Season'], game['DayNum'],
                             winner_team_id, loser_team_id, label=1, team_a_location=team_a_location,
                             is_tournament=game['is_tournament'])
    else:
        location_flip = {'H': 'A', 'A': 'H', 'N': 'N'}
        team_a_location = location_flip[game['WLoc']]
        row = featurize_game(loser_entry, winner_entry, game['Season'], game['DayNum'],
                             loser_team_id, winner_team_id, label=0, team_a_location=team_a_location,
                             is_tournament=game['is_tournament'])
    if row is not None:
        training_rows.append(row)

    # --- Adjusted efficiency update ---
    # expected = own_off + opp_def - league_avg; update ratings by k * (raw - expected)
    possessions = (estimate_possessions(game, 'W') + estimate_possessions(game, 'L')) / 2

    if possessions > 0:
        winner_raw_offensive = game['WScore'] / possessions * 100.0
        loser_raw_offensive = game['LScore'] / possessions * 100.0

        # Home court adjustment: remove HCA from raw efficiencies
        if game['WLoc'] == 'H':
            winner_raw_offensive -= HOME_COURT_ADVANTAGE
            loser_raw_offensive += HOME_COURT_ADVANTAGE
        elif game['WLoc'] == 'A':
            winner_raw_offensive += HOME_COURT_ADVANTAGE
            loser_raw_offensive -= HOME_COURT_ADVANTAGE
        # 'N': no adjustment

        league_average = get_league_average_efficiency(game['Season'])

        winner_expected_offensive = (winner_entry['offensive_efficiency']
                                     + loser_entry['defensive_efficiency']
                                     - league_average)
        loser_expected_offensive = (loser_entry['offensive_efficiency']
                                    + winner_entry['defensive_efficiency']
                                    - league_average)

        # Residual capping to limit impact of outlier games
        winner_offensive_residual = np.clip(winner_raw_offensive - winner_expected_offensive, -RESIDUAL_CAP, RESIDUAL_CAP)
        loser_offensive_residual = np.clip(loser_raw_offensive - loser_expected_offensive, -RESIDUAL_CAP, RESIDUAL_CAP)

        winner_entry['offensive_efficiency'] += EFFICIENCY_LEARNING_RATE * winner_offensive_residual
        loser_entry['defensive_efficiency'] += EFFICIENCY_LEARNING_RATE * winner_offensive_residual
        loser_entry['offensive_efficiency'] += EFFICIENCY_LEARNING_RATE * loser_offensive_residual
        winner_entry['defensive_efficiency'] += EFFICIENCY_LEARNING_RATE * loser_offensive_residual

    # Accumulate league totals
    season_totals = get_or_create_league_totals(game['Season'])
    season_totals['total_points'] += game['WScore'] + game['LScore']
    season_totals['total_possessions'] += 2 * possessions
    season_totals['game_count'] += 1

    # Accumulate team possessions
    winner_entry['total_possessions'] += possessions
    loser_entry['total_possessions'] += possessions

    update_team_box_stats(winner_entry, game, 'W', 'L')
    update_team_box_stats(loser_entry, game, 'L', 'W')

print(f"Processed {len(results)} games, {len(team_rolling)} team-seasons")
print(f"Training rows collected: {len(training_rows)}")

In [20]:
training_data = pd.DataFrame(training_rows)
print(f"Training rows: {training_data.shape[0]}, Features: {training_data.shape[1] - 6}")
print(f"Label balance: {training_data['label'].mean():.3f}")
print(f"Location values: {sorted(training_data['team_a_location'].unique())}")
display(training_data.head())

Training rows: 118914, Features: 120
Label balance: 0.488
Location values: ['A', 'H', 'N']


,team_a_avg_score,team_a_avg_opponent_score,team_a_avg_fgm,team_a_avg_opponent_fgm,team_a_avg_fga,team_a_avg_opponent_fga,team_a_avg_fgm3,team_a_avg_opponent_fgm3,team_a_avg_fga3,team_a_avg_opponent_fga3,...,diff_offensive_efficiency,diff_defensive_efficiency,diff_net_efficiency,diff_avg_possessions,season,day_number,team_a_id,team_b_id,label,team_a_location
0,55.0,81.0,20.0,26.0,46.0,57.0,3.0,6.0,11.0,12.0,...,0.010417,1.921303,-1.910885,8.5250,2003,12,1186,1457,1,N
1,56.0,50.0,18.0,18.0,38.0,49.0,3.0,6.0,9.0,22.0,...,-1.921303,-0.010417,-1.910885,-8.5250,2003,12,1296,1458,0,A
2,48.0,76.0,18.0,25.0,64.0,56.0,8.0,10.0,24.0,23.0,...,-1.037063,0.609875,-1.646939,3.2375,2003,14,1125,1135,1,N
3,66.0,71.0,24.0,28.0,52.0,58.0,6.0,5.0,18.0,11.0,...,0.207130,-0.749583,0.956713,-0.7250,2003,14,1156,1236,1,N
4,80.0,62.0,23.0,19.0,55.0,41.0,2.0,4.0,8.0,15.0,...,0.749583,-0.207130,0.956713,0.7250,2003,14,1161,1194,1,H


In [21]:
rows = []
for (season, team_id), entry in team_rolling.items():
    row = {'Season': season, 'TeamID': team_id, 'game_count': entry['game_count']}
    for stat_name, total in entry['sums'].items():
        row[f'avg_{stat_name.lower()}'] = total / entry['game_count']
    row['offensive_efficiency'] = entry['offensive_efficiency']
    row['defensive_efficiency'] = entry['defensive_efficiency']
    row['net_efficiency'] = entry['offensive_efficiency'] - entry['defensive_efficiency']
    row['avg_possessions'] = entry['total_possessions'] / entry['game_count']
    rows.append(row)

team_season_stats = pd.DataFrame(rows).sort_values(['Season', 'TeamID']).reset_index(drop=True)

def safe_rate(numerator, denominator):
    """Compute numerator / denominator, returning 0.0 where denominator is zero."""
    return np.where(denominator == 0, 0.0, numerator / denominator)

# Rate stats (from season totals — properly weighted)
team_season_stats['fg_pct'] = safe_rate(team_season_stats['avg_fgm'], team_season_stats['avg_fga'])
team_season_stats['fg3_pct'] = safe_rate(team_season_stats['avg_fgm3'], team_season_stats['avg_fga3'])
team_season_stats['ft_pct'] = safe_rate(team_season_stats['avg_ftm'], team_season_stats['avg_fta'])
team_season_stats['fg3_rate'] = safe_rate(team_season_stats['avg_fga3'], team_season_stats['avg_fga'])

# Same for opponent (defensive) rates
team_season_stats['opponent_fg_pct'] = safe_rate(team_season_stats['avg_opponent_fgm'], team_season_stats['avg_opponent_fga'])
team_season_stats['opponent_fg3_pct'] = safe_rate(team_season_stats['avg_opponent_fgm3'], team_season_stats['avg_opponent_fga3'])
team_season_stats['opponent_ft_pct'] = safe_rate(team_season_stats['avg_opponent_ftm'], team_season_stats['avg_opponent_fta'])
team_season_stats['opponent_fg3_rate'] = safe_rate(team_season_stats['avg_opponent_fga3'], team_season_stats['avg_opponent_fga'])

display(team_season_stats)

,Season,TeamID,game_count,avg_score,avg_opponent_score,avg_fgm,avg_opponent_fgm,avg_fga,avg_opponent_fga,avg_fgm3,...,net_efficiency,avg_possessions,fg_pct,fg3_pct,ft_pct,fg3_rate,opponent_fg_pct,opponent_fg3_pct,opponent_ft_pct,opponent_fg3_rate
0,2003,1102,28,57.250000,57.000000,19.142857,19.285714,39.785714,42.428571,7.821429,...,-1.255548,55.045536,0.481149,0.375643,0.651357,0.523339,0.454545,0.382184,0.710575,0.292929
1,2003,1103,27,78.777778,78.148148,27.148148,27.777778,55.851852,57.000000,5.444444,...,-1.475007,70.900000,0.486074,0.338710,0.736390,0.287798,0.487329,0.362903,0.719064,0.322287
2,2003,1104,28,69.285714,65.000000,24.035714,23.250000,57.178571,55.500000,6.357143,...,9.722408,66.720536,0.420362,0.320144,0.709898,0.347283,0.418919,0.332090,0.708333,0.344916
3,2003,1105,26,71.769231,76.653846,24.384615,27.000000,61.615385,58.961538,7.576923,...,-8.438848,76.680288,0.395755,0.364815,0.705986,0.337079,0.457926,0.357456,0.668760,0.297456
4,2003,1106,28,63.607143,63.750000,23.428571,21.714286,55.285714,53.392857,6.107143,...,-6.894017,67.716071,0.423773,0.346154,0.646421,0.319121,0.406689,0.314554,0.707317,0.284950
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8341,2026,1477,29,66.931034,75.758621,23.344828,26.724138,56.000000,58.620690,8.034483,...,-12.625749,68.637500,0.416872,0.310253,0.687379,0.462438,0.455882,0.373065,0.692833,0.380000
8342,2026,1478,30,72.366667,73.666667,24.733333,25.333333,53.800000,60.133333,7.566667,...,-5.609181,68.551667,0.459727,0.344985,0.716511,0.407683,0.421286,0.358974,0.737374,0.389135
8343,2026,1479,29,68.551724,69.172414,25.896552,23.724138,58.206897,52.931034,5.793103,...,-5.967569,65.568103,0.444905,0.316981,0.734411,0.313981,0.448208,0.332046,0.705701,0.337459
8344,2026,1480,28,75.142857,80.678571,27.678571,28.571429,62.607143,60.000000,6.892857,...,-8.379016,71.189286,0.442099,0.335652,0.699612,0.328009,0.476190,0.335505,0.730645,0.365476
